### Week 5 Day 4

AutoGen Core - Distributed

I'm only going to give a Teaser of this!!

Partly because I'm unsure how relevant it is to you. If you'd like me to add more content for this, please do let me know..

In [1]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool
from IPython.display import display, Markdown

from dotenv import load_dotenv

load_dotenv(override=True)

ALL_IN_ONE_WORKER = False

### Start with our Message class

In [2]:

@dataclass
class Message:
    content: str

### And now - a host for our distributed runtime

In [3]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost

host = GrpcWorkerAgentRuntimeHost(address="localhost:50051")
host.start() 

### Let's reintroduce a tool

In [4]:
serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="Useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

In [5]:
instruction1 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons in favor of choosing AutoGen; the pros of AutoGen."

instruction2 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons against choosing AutoGen; the cons of Autogen."

judge = "You must make a decision on whether to use AutoGen for a project. \
Your research team has come up with the following reasons for and against. \
Based purely on the research from your team, please respond with your decision and brief rationale."

### And make some Agents

In [6]:
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Judge(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        message1 = Message(content=instruction1)
        message2 = Message(content=instruction2)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        result = f"## Pros of AutoGen:\n{response1.content}\n\n## Cons of AutoGen:\n{response2.content}\n\n"
        judgement = f"{judge}\n{result}Respond with your decision and brief explanation"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)


In [7]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()

    await Player1Agent.register(worker, "player1", lambda: Player1Agent("player1"))
    await Player2Agent.register(worker, "player2", lambda: Player2Agent("player2"))
    await Judge.register(worker, "judge", lambda: Judge("judge"))

    agent_id = AgentId("judge", "default")

else:

    worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker1.start()
    await Player1Agent.register(worker1, "player1", lambda: Player1Agent("player1"))

    worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker2.start()
    await Player2Agent.register(worker2, "player2", lambda: Player2Agent("player2"))

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    agent_id = AgentId("judge", "default")




In [8]:
response = await worker.send_message(Message(content="Go!"), agent_id)

In [9]:
display(Markdown(response.content))

## Pros of AutoGen:
Here are some reasons to consider using AutoGen for your AI Agent project:

1. **Rapid Prototyping**: AutoGen allows teams to transition quickly from concept to prototype. Its low-code interface enables developers to visualize and construct applications efficiently.

2. **Ease of Use**: It offers built-in agents that can be utilized directly, making it accessible for developers even if they lack extensive experience with AI or coding.

3. **Integration Capabilities**: AutoGen facilitates seamless integration between large language models (LLMs), various tools, and human inputs, enhancing the collaborative nature of task execution.

4. **Multi-Agent Collaboration**: The framework supports the development of agent teams that can work together to solve complex problems dynamically, which is essential for intricate projects requiring multiple perspectives.

5. **Open Source**: Being an open-source framework, AutoGen provides flexibility and the potential for customization, allowing developers to tailor it to their specific needs.

6. **Efficiency in Development**: AutoGen manages the complexities of AI application development, enabling teams to focus on creative and functional aspects of their projects rather than getting mired in technical details.

7. **Support for Competitive Advantage**: By streamlining the development process, AutoGen helps teams build superior applications faster, thereby maintaining a competitive edge in the market.

Overall, these advantages make AutoGen a compelling choice for teams looking to develop AI agents effectively and efficiently. 

TERMINATE

## Cons of AutoGen:
Here are some cons of using AutoGen for an AI Agent project:

1. **Steep Learning Curve**: AutoGen requires knowledge of Python programming, LLM APIs, and multi-agent concepts, which can be challenging for developers who are new to these areas.

2. **Complex Debugging**: Users have reported that debugging complex agent interactions can be difficult and requires a deep technical understanding, potentially leading to longer development times.

3. **Limited Robustness**: There may be issues with the robustness of the agents in unpredictable real-world scenarios, limiting their effectiveness in certain applications.

4. **Documentation and Support**: While there is some documentation available, comprehensive resources may be lacking, making it harder for developers to find answers during implementation.

5. **Integration Challenges**: Integrating AutoGen with existing systems or workflows may pose challenges, requiring additional development effort.

These considerations could be significant depending on your project's needs and the skill level of your development team. TERMINATE



## Decision:

Based on the research provided by your team, I recommend using AutoGen for the project. The advantages outweigh the disadvantages, particularly the benefits of rapid prototyping, ease of use, and multi-agent collaboration. These factors are essential for effective AI agent development and can significantly enhance team efficiency and output quality.

While the learning curve and debugging complexities are valid concerns, the overall flexibility and open-source nature of AutoGen provide opportunities for customization and growth. If the team is willing to invest time in training and development, the long-term benefits of using AutoGen seem to support project goals effectively.

TERMINATE

In [10]:
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await worker1.stop()
    await worker2.stop()

In [11]:
await host.stop()